In [1]:
import sys
from pathlib import Path

# Add parent directory of "a" to sys.path
sys.path.append(str(Path.cwd().parent))

In [2]:
import numpy as np
import pandas as pd
from CODECbreakCode.AudioMixer import SingleFileAudioMixer
import CODECbreakCode.Evaluator as Evaluator
from CODECbreakCode.Evaluator import MeasureHAAQIOutput
import argparse
from CODECbreakCode.Optimiser.config import get_config, normalize_action, denormalize_action

In [3]:
import warnings
warnings.filterwarnings("ignore", "Possible clipped samples in output.")
warnings.filterwarnings("ignore",message="Warning: input samples dtype is np.float64. Converting to np.float32")

## Male English

In [4]:
Speech_Mixing_Path = str(Path("../AudioEX/Speech").resolve())
Speech_Noise_Generator_MP3 = SingleFileAudioMixer(Speech_Mixing_Path,"Male_English.wav")
Speech_Referece_File,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack([0]*7,"Reference_IN_MaleEng.wav")
print(f"Referece_File:{Speech_Referece_File}")
Referece_MP3File = Evaluator.Mp3LameLossyCompress(Speech_Referece_File,64)
print(f"Reggae_Referece_Mp3File:{Referece_MP3File}")

Referece_File:/home/codecrack/CodecBreakerwithRL/Paper_Code_Framework_for_Degradator/AudioEX/Speech/Mixing_Result/Reference_IN_MaleEng.wav
Reggae_Referece_Mp3File:/home/codecrack/CodecBreakerwithRL/Paper_Code_Framework_for_Degradator/AudioEX/Speech/Mixing_Result_Mp3_Wav/Reference_IN_MaleEng_64kbps.wav


In [5]:
####initalise the Haaqi
MeasureHAAQI = MeasureHAAQIOutput(Referece_MP3File)#Initilize the HAAQI with a permanent reference
MeasureHAAQI.MeasureHAQQIOutput(Referece_MP3File) #Test on how far from itself to itself

0.9989990004761835

In [6]:
import tempfile
import os
BASE_DIR = Speech_Mixing_Path + "/Mixing_Result"
TMP_SUBDIR = os.path.join(BASE_DIR, "tmp")

# make sure it exists once at startup:
os.makedirs(TMP_SUBDIR, exist_ok=True)
def haaqi_reward_muti_fn(solution: np.ndarray, is_normalised=True) -> float:
    if is_normalised:
        solution = denormalize_action(solution)
#    print(f'solution:{solution}, the type is {type(solution), type(solution[0])}')
 
    # Create a unique temp‐file name
    fd, degradated_filename = tempfile.mkstemp(prefix="dynC_", suffix=".wav")
    os.close(fd)  # we’ll let your compressor write to that path
    #print(f"degraded File:{degradated_filename}")
    
    try:
        solution = list(solution)
        gener_Audio,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack(solution,degradated_filename)
        gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(gener_Audio, 64)
        score = MeasureHAAQI.MeasureHAQQIOutput(gener_Audio_mp3)
    finally:
        # clean up
        try:
            os.remove(degradated_filename)
        except OSError:
            pass

    return round(1 - score, 4)

#### GA

In [ ]:
from CODECbreakCode.Optimiser.genetic_evo_ind import GeneticOptimiser_Ind

In [ ]:
GA_Opt = GeneticOptimiser_Ind()
GA_Opt.ga_init_env()
GA_Opt.set_fitnessfun(haaqi_reward_muti_fn)

In [ ]:
GA_Opt.run(num_generations = 10,sol_per_pop = 60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
Evo_Data, Evo_Data_Full = GA_Opt.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

#### MC

In [ ]:
from CODECbreakCode.Optimiser.mc_opt_multi import MonteCarloMultiOptimiser
from CODECbreakCode.Optimiser.config import get_single_config
cfg = get_single_config()

In [ ]:
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=7, low = cfg["env"]["act_min"],high = cfg["env"]["act_min"])
per_step_bests = mc.optimise_by_steps(n_steps=1, sampling_per_step=2)

In [ ]:
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=7,low = cfg["env"]["act_min"],high = cfg["env"]["act_min"])
per_step_bests = mc.optimise_by_steps(n_steps=10, sampling_per_step=60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
MC_Data, MC_Data_Full = mc.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

#### RL

In [7]:
from CODECbreakCode.Optimiser.continous_RL_train_Ind_audio import continous_RL_train_ind_audio as CRLTrain

2026-05-26 15:55:09.327725: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-26 15:55:09.358659: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-26 15:55:09.358689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-26 15:55:09.362615: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-26 15:55:09.374601: I tensorflow/core/platform/cpu_feature_guar

In [8]:
trainner = CRLTrain(sub_episode_length=16, sub_episode_num_single_batch=4, env_num=1)
trainner.set_environments(haaqi_reward_muti_fn)
trainner.train(update_num=10, eval_intv=5)

number of sub_episodes used for a single param update: 4
train_env.batch_size = parallel environment number =  1
update_num: 10, eval_intv: 5
Instructions for updating:
Use `as_dataset(..., single_deterministic_pass=True)` instead.
[step   0] mean entropy per-dim = 38.7943


InvalidArgumentError: {{function_node __wrapped__Sub_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes: [28] vs. [7] [Op:Sub] name: 

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
Evo_Data, Evo_Data_Full = trainner.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
Speech_Mixing_Path = '/home/codecrack/CodecBreakerwithRL/AudioEX/Speech'
Speech_Noise_Generator_MP3 = SingleFileAudioMixer(Speech_Mixing_Path,"FeMale_English.wav")
Speech_Referece_File,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack([0]*7,"Reference_IN_FeMaleEng.wav")
print(f"Referece_File:{Speech_Referece_File}")
Referece_MP3File = Evaluator.Mp3LameLossyCompress(Speech_Referece_File,64)
print(f"Reggae_Referece_Mp3File:{Referece_MP3File}")

In [ ]:
####initalise the Haaqi
MeasureHAAQI = MeasureHAAQIOutput(Referece_MP3File)#Initilize the HAAQI with a permanent reference
MeasureHAAQI.MeasureHAQQIOutput(Referece_MP3File) #Test on how far from itself to itself

In [ ]:
import tempfile
import os
BASE_DIR = "/home/codecrack/CodecBreakerwithRL/AudioEX/Speech/Mixing_Result"
TMP_SUBDIR = os.path.join(BASE_DIR, "tmp")

# make sure it exists once at startup:
os.makedirs(TMP_SUBDIR, exist_ok=True)
def haaqi_reward_muti_fn(solution: np.ndarray, is_normalised=True) -> float:
    if is_normalised:
        solution = denormalize_action(solution)
    #print(f'solution:{solution}, the type is {type(solution), type(solution[0])}')
 
    # Create a unique temp‐file name
    fd, degradated_filename = tempfile.mkstemp(prefix="dynC_", suffix=".wav")
    os.close(fd)  # we’ll let your compressor write to that path
    #print(f"degraded File:{degradated_filename}")
    
    try:
        solution = list(solution)
        gener_Audio,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack(solution,degradated_filename)
        gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(gener_Audio, 64)
        score = MeasureHAAQI.MeasureHAQQIOutput(gener_Audio_mp3)
    finally:
        # clean up
        try:
            os.remove(degradated_filename)
        except OSError:
            pass

    return round(1 - score, 2)

In [ ]:
from Optimiser.genetic_evo_ind import GeneticOptimiser_Ind
GA_Opt = GeneticOptimiser_Ind()
GA_Opt.ga_init_env()
GA_Opt.set_fitnessfun(haaqi_reward_muti_fn)
GA_Opt.run(num_generations = 10,sol_per_pop = 60)


In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
Evo_Data, Evo_Data_Full = GA_Opt.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
from Optimiser.mc_opt_multi import MonteCarloMultiOptimiser
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=7,low =[40, 40, 0.1, -25.0, 1.0, 1.0, 100.0],high = [60, 60, 3.0, 0.0, 5.0, 10.0, 300.0],)
per_step_bests = mc.optimise_by_steps(n_steps=10, sampling_per_step=60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
MC_Data, MC_Data_Full = mc.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
Speech_Mixing_Path = '/home/codecrack/CodecBreakerwithRL/AudioEX/Speech'
Speech_Noise_Generator_MP3 = SingleFileAudioMixer(Speech_Mixing_Path,"MusicWithSpeech.wav")
Speech_Referece_File,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack([0]*7,"Reference_IN_MusicwithSpeech.wav")
print(f"Referece_File:{Speech_Referece_File}")
Referece_MP3File = Evaluator.Mp3LameLossyCompress(Speech_Referece_File,64)
print(f"Reggae_Referece_Mp3File:{Referece_MP3File}")
####initalise the Haaqi
MeasureHAAQI = MeasureHAAQIOutput(Referece_MP3File)#Initilize the HAAQI with a permanent reference
MeasureHAAQI.MeasureHAQQIOutput(Referece_MP3File) #Test on how far from itself to itself

In [ ]:
import tempfile
import os
BASE_DIR = "/home/codecrack/CodecBreakerwithRL/AudioEX/Speech/Mixing_Result"
TMP_SUBDIR = os.path.join(BASE_DIR, "tmp")

# make sure it exists once at startup:
os.makedirs(TMP_SUBDIR, exist_ok=True)
def haaqi_reward_muti_fn(solution: np.ndarray, is_normalised=True) -> float:
    if is_normalised:
        solution = denormalize_action(solution)
    #print(f'solution:{solution}, the type is {type(solution), type(solution[0])}')
 
    # Create a unique temp‐file name
    fd, degradated_filename = tempfile.mkstemp(prefix="dynC_", suffix=".wav")
    os.close(fd)  # we’ll let your compressor write to that path
    #print(f"degraded File:{degradated_filename}")
    
    try:
        solution = list(solution)
        gener_Audio,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack(solution,degradated_filename)
        gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(gener_Audio, 64)
        score = MeasureHAAQI.MeasureHAQQIOutput(gener_Audio_mp3)
    finally:
        # clean up
        try:
            os.remove(degradated_filename)
        except OSError:
            pass

    return round(1 - score, 3)

In [ ]:
from Optimiser.genetic_evo_ind import GeneticOptimiser_Ind
GA_Opt = GeneticOptimiser_Ind()
GA_Opt.ga_init_env()
GA_Opt.set_fitnessfun(haaqi_reward_muti_fn)
GA_Opt.run(num_generations = 10,sol_per_pop = 60)


In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
Evo_Data, Evo_Data_Full = GA_Opt.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
from Optimiser.mc_opt_multi import MonteCarloMultiOptimiser
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=7,low =[30, 30, 0.0, -10.0, 1.0, 0.1, 50],high = [60, 60, 5.0, 0.0, 10.0, 10.0, 300.0])
per_step_bests = mc.optimise_by_steps(n_steps=10, sampling_per_step=60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
MC_Data, MC_Data_Full = mc.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
Speech_Mixing_Path = '/home/codecrack/CodecBreakerwithRL/AudioEX/Speech'
Speech_Noise_Generator_MP3 = SingleFileAudioMixer(Speech_Mixing_Path,"Male_German.wav")
Speech_Referece_File,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack([0]*7,"Reference_IN_Male_German.wav")
print(f"Referece_File:{Speech_Referece_File}")
Referece_MP3File = Evaluator.Mp3LameLossyCompress(Speech_Referece_File,64)
print(f"Reggae_Referece_Mp3File:{Referece_MP3File}")
####initalise the Haaqi
MeasureHAAQI = MeasureHAAQIOutput(Referece_MP3File)#Initilize the HAAQI with a permanent reference
MeasureHAAQI.MeasureHAQQIOutput(Referece_MP3File) #Test on how far from itself to itself

In [ ]:
import tempfile
import os
BASE_DIR = "/home/codecrack/CodecBreakerwithRL/AudioEX/Speech/Mixing_Result"
TMP_SUBDIR = os.path.join(BASE_DIR, "tmp")

# make sure it exists once at startup:
os.makedirs(TMP_SUBDIR, exist_ok=True)
def haaqi_reward_muti_fn(solution: np.ndarray, is_normalised=True) -> float:
    if is_normalised:
        solution = denormalize_action(solution)
    #print(f'solution:{solution}, the type is {type(solution), type(solution[0])}')
 
    # Create a unique temp‐file name
    fd, degradated_filename = tempfile.mkstemp(prefix="dynC_", suffix=".wav")
    os.close(fd)  # we’ll let your compressor write to that path
    #print(f"degraded File:{degradated_filename}")
    
    try:
        solution = list(solution)
        gener_Audio,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack(solution,degradated_filename)
        gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(gener_Audio, 64)
        score = MeasureHAAQI.MeasureHAQQIOutput(gener_Audio_mp3)
    finally:
        # clean up
        try:
            os.remove(degradated_filename)
        except OSError:
            pass

    return round(1 - score, 3)

In [ ]:
from Optimiser.genetic_evo_ind import GeneticOptimiser_Ind
GA_Opt = GeneticOptimiser_Ind()
GA_Opt.ga_init_env()
GA_Opt.set_fitnessfun(haaqi_reward_muti_fn)
GA_Opt.run(num_generations = 10,sol_per_pop = 60)


In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
Evo_Data, Evo_Data_Full = GA_Opt.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
from Optimiser.mc_opt_multi import MonteCarloMultiOptimiser
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=7,low =[30, 30, 0.0, -10.0, 1.0, 0.1, 50],high = [60, 60, 5.0, 0.0, 10.0, 10.0, 300.0])
per_step_bests = mc.optimise_by_steps(n_steps=10, sampling_per_step=60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
MC_Data, MC_Data_Full = mc.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
Speech_Mixing_Path = '/home/codecrack/CodecBreakerwithRL/AudioEX/Speech'
Speech_Noise_Generator_MP3 = SingleFileAudioMixer(Speech_Mixing_Path,"Male_German_short.wav")
Speech_Referece_File,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack([0]*7,"Reference_IN_Male_German_short.wav")
print(f"Referece_File:{Speech_Referece_File}")
Referece_MP3File = Evaluator.Mp3LameLossyCompress(Speech_Referece_File,64)
print(f"Reggae_Referece_Mp3File:{Referece_MP3File}")
####initalise the Haaqi
MeasureHAAQI = MeasureHAAQIOutput(Referece_MP3File)#Initilize the HAAQI with a permanent reference
MeasureHAAQI.MeasureHAQQIOutput(Referece_MP3File) #Test on how far from itself to itself

In [ ]:
import tempfile
import os
BASE_DIR = "/home/codecrack/CodecBreakerwithRL/AudioEX/Speech/Mixing_Result"
TMP_SUBDIR = os.path.join(BASE_DIR, "tmp")

# make sure it exists once at startup:
os.makedirs(TMP_SUBDIR, exist_ok=True)
def haaqi_reward_muti_fn(solution: np.ndarray, is_normalised=True) -> float:
    if is_normalised:
        solution = denormalize_action(solution)
    #print(f'solution:{solution}, the type is {type(solution), type(solution[0])}')
 
    # Create a unique temp‐file name
    fd, degradated_filename = tempfile.mkstemp(prefix="dynC_", suffix=".wav")
    os.close(fd)  # we’ll let your compressor write to that path
    #print(f"degraded File:{degradated_filename}")
    
    try:
        solution = list(solution)
        gener_Audio,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack(solution,degradated_filename)
        gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(gener_Audio, 64)
        score = MeasureHAAQI.MeasureHAQQIOutput(gener_Audio_mp3)
    finally:
        # clean up
        try:
            os.remove(degradated_filename)
        except OSError:
            pass

    return round(1 - score, 2)

In [ ]:
from Optimiser.genetic_evo_ind import GeneticOptimiser_Ind
GA_Opt = GeneticOptimiser_Ind()
GA_Opt.ga_init_env()
GA_Opt.set_fitnessfun(haaqi_reward_muti_fn)
GA_Opt.run(num_generations = 10,sol_per_pop = 60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
Evo_Data, Evo_Data_Full = GA_Opt.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

In [ ]:
from Optimiser.mc_opt_multi import MonteCarloMultiOptimiser
mc = MonteCarloMultiOptimiser(reward_fn = haaqi_reward_muti_fn,solution_dim=7,low =[40, 40, 0.1, -25.0, 1.0, 1.0, 100.0],high = [60, 60, 3.0, 0.0, 5.0, 10.0, 300.0],)
per_step_bests = mc.optimise_by_steps(n_steps=10, sampling_per_step=60)

In [ ]:
para_columns=['Hum','Hiss','CL','Thre','Ratio','Attk','Release',]
MC_Data, MC_Data_Full = mc.save_results(filefold=Speech_Mixing_Path, para_columns=para_columns, is_outputfulldata = True)

## TEST Cross Exam

In [ ]:
list([54.59,50.87,2.81,-4.6,1.01,8.72,106.72])

In [ ]:
Deg_Speech_Referece_File,_ = Speech_Noise_Generator_MP3.TestDynNoisedFullTrack([54.59,50.87,2.81,-4.6,1.01,8.72,106.72],"Deg_IN_MaleEng.wav")
Deg_gener_Audio_mp3 = Evaluator.Mp3LameLossyCompress(Deg_Speech_Referece_File, 64)
1-MeasureHAAQI.MeasureHAQQIOutput(Deg_gener_Audio_mp3)